In [ ]:
import os
import glob
import pandas as pd

RESULTS_DIRECTORY = '../results'

# Recursively find all .csv files in subdirectories under RESULTS_DIRECTORY
csv_file_list = glob.glob(os.path.join(RESULTS_DIRECTORY, '**', '*.csv'), recursive=True)

# Read all CSVs into a list of DataFrames
dfs = [pd.read_csv(f) for f in csv_file_list]

# Concatenate all DataFrames
concatenated_df = pd.concat(dfs, ignore_index=True)

# Save as concatenated_results.csv for further processing
concatenated_df.to_csv('concatenated_results.csv', index=False)

# Now concatenated_df is ready for further processing

In [ ]:
concatenated_df.head(5)

In [ ]:
# Extract experiment fields from lif_container_id
experiment_fields = concatenated_df['lif_container_id'].str.split('_', expand=True)
concatenated_df['experiment_date'] = experiment_fields[0]
concatenated_df['experiment_name'] = experiment_fields[1]
concatenated_df['treatment'] = experiment_fields[2]
concatenated_df['experiment_id'] = experiment_fields[0] + '_' + experiment_fields[1]

# Extract genotype and replica_id from lif_image_name
image_fields = concatenated_df['lif_image_name'].str.split(' ', expand=True)
concatenated_df['genotype'] = image_fields[0]
concatenated_df['replica_id'] = image_fields[1]

# Reorder columns: keep existing ID columns once, then insert only derived fields
cols = list(concatenated_df.columns)
li_index = cols.index('lif_image_name')
front = cols[:li_index + 1]
derived = ['experiment_date', 'experiment_name', 'treatment', 'experiment_id', 'genotype', 'replica_id']
new_order = front + derived + [c for c in cols if c not in (front + derived)]
concatenated_df = concatenated_df[new_order]

# Save the updated DataFrame, overwriting the file
concatenated_df.to_csv('concatenated_results.csv', index=False)

concatenated_df.head(5)

In [ ]:
# Group by 'lif_container_id', 'lif_image_name', 'tissue_layer' and keep non-numeric (non-aggregated) columns
# Directly derived non-numeric columns to retain
non_numeric_columns = [
    'experiment_date', 'experiment_name', 'treatment', 'experiment_id', 'genotype', 'replica_id'
]

# Numeric columns for aggregation
numeric_columns = concatenated_df.select_dtypes(include='number').columns.tolist()

# Combine for aggregation: use 'mean' for numerics, 'first' for non-numerics
agg_dict = {col: 'mean' for col in numeric_columns}
agg_dict.update({col: 'first' for col in non_numeric_columns if col in concatenated_df.columns})

grouped_df = (
    concatenated_df
    .groupby(['lif_container_id', 'lif_image_name', 'tissue_layer'], as_index=False)
    .agg(agg_dict)
)

grouped_df.to_csv('concatenated_results_mean.csv', index=False)

grouped_df.head(20)

In [ ]:
grouped_df.columns

In [ ]:
columns_to_plot = ['area', 'FRET_ratio_sum_norm_per_image', 'FRET_ratio_mean_norm_per_image', 'FRET_ratio_sum', 'FRET_ratio_mean']

In [ ]:
import plotly.express as px

for column in columns_to_plot:
    fig = px.scatter(
        grouped_df,
        x='tissue_layer',
        y=column,
        color='treatment',  # Color code by treatment
        hover_data=['lif_container_id', 'lif_image_name', 'area', 'treatment'],
        title=f"{column} by Tissue Layer (Colored by Treatment)",
    )
    fig.show()